# Add Google Gemini LLM Provider

This notebook is a planning and verification guide for adding a Google Gemini implementation of `src.llm.providers.base.LLMProvider`.

The goal is to prove the provider behavior before adding production code:

1. Authenticate with `google-genai`.
2. Convert InnomightLabs messages into Gemini `Content` objects.
3. Stream text chunks as `LLMEvent(type="text")`.
4. Convert generic function tools into Gemini function declarations.
5. Stream Gemini function calls as `LLMEvent(type="tool_use")`.
6. Send a tool result back to Gemini and receive the final answer.

Keep secrets out of the notebook. Use environment variables before launching Jupyter.

## Prerequisites

Run the notebook with the API virtual environment as the kernel if possible. The `google-genai` package is expected to be installed in `api/.venv` by `uv`.

Recommended local launch:

```bash
cd api
uv run jupyter lab ../developer-manual/add-llm-provider.ipynb
```

For Google AI Studio / Gemini Developer API authentication, set:

```bash
export GEMINI_API_KEY='<gemini-api-key>'
```

For OAuth credentials, `client_id` and `client_secret` alone are not enough to call Gemini. The notebook supports OAuth only when you also provide an access token or refresh token:

```bash
export GEMINI_CLIENT_ID='<oauth-client-id>'
export GEMINI_CLIENT_SECRET='<oauth-client-secret>'
export GEMINI_REFRESH_TOKEN='<oauth-refresh-token>'
# optional if you already have a live access token
export GEMINI_ACCESS_TOKEN='<oauth-access-token>'
```

Optional model override:

```bash
export GEMINI_MODEL='gemini-2.5-flash'
```


In [1]:
import dotenv
import asyncio
import json
import os
import sys
from dataclasses import asdict
from pathlib import Path
from typing import Any, AsyncIterator

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "api":
    API_ROOT = PROJECT_ROOT
elif (PROJECT_ROOT / "api").exists():
    API_ROOT = PROJECT_ROOT / "api"
else:
    API_ROOT = PROJECT_ROOT.parent / "api"

sys.path.insert(0, str(API_ROOT))

from google import genai
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google.genai import types

from src.llm.providers.base import LLMEvent

DEFAULT_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")

print("API_ROOT:", API_ROOT)
print("DEFAULT_MODEL:", DEFAULT_MODEL)


API_ROOT: /Users/vslala/src/code/projects/innomightlabs/innomightlabs-prod/api
DEFAULT_MODEL: gemini-3.5-flash


## Step 1: Build A Gemini Client

The production provider should accept a `credentials` dict. The first implementation should support `{"api_key": "..."}` because `google-genai` can call the Gemini Developer API directly with an API key.

OAuth can be added generically with `{"client_id", "client_secret", "refresh_token"}` or `{"access_token"}`. A client ID and client secret without a token cannot authenticate API calls.

In [2]:
env_file = ".env"
dotenv.load_dotenv(env_file, override=True)

def load_notebook_credentials() -> dict[str, str]:
    credentials = {
        "api_key": os.environ.get("GEMINI_API_KEY", "").strip(),
        "client_id": os.getenv("GEMINI_CLIENT_ID", "").strip(),
        "client_secret": os.getenv("GEMINI_CLIENT_SECRET", "").strip(),
        "access_token": os.getenv("GEMINI_ACCESS_TOKEN", "").strip(),
        "refresh_token": os.getenv("GEMINI_REFRESH_TOKEN", "").strip(),
    }
    return {key: value for key, value in credentials.items() if value}


def build_gemini_client(credentials: dict[str, str]) -> genai.Client:
    api_key = credentials.get("api_key")
    if api_key:
        return genai.Client(api_key=api_key)

    client_id = credentials.get("client_id")
    client_secret = credentials.get("client_secret")
    access_token = credentials.get("access_token")
    refresh_token = credentials.get("refresh_token")
    if client_id and client_secret and (access_token or refresh_token):
        oauth_credentials = Credentials(
            token=access_token or None,
            refresh_token=refresh_token or None,
            token_uri="https://oauth2.googleapis.com/token",
            client_id=client_id,
            client_secret=client_secret,
            scopes=["https://www.googleapis.com/auth/generative-language"],
        )
        if not oauth_credentials.valid and oauth_credentials.refresh_token:
            oauth_credentials.refresh(Request())
        return genai.Client(credentials=oauth_credentials)

    raise RuntimeError(
        "Set GEMINI_API_KEY, or set GEMINI_CLIENT_ID/GEMINI_CLIENT_SECRET plus "
        "GEMINI_ACCESS_TOKEN or GEMINI_REFRESH_TOKEN."
    )


notebook_credentials = load_notebook_credentials()
client = build_gemini_client(notebook_credentials)
print("Credential mode:", "api_key" if "api_key" in notebook_credentials else "oauth")


Credential mode: api_key


## Step 2: Convert InnomightLabs Messages

`LLMProvider.stream_response()` receives messages as dicts with `role` and `content`.

Provider mapping:

- `system` messages become `GenerateContentConfig.system_instruction`.
- `user` messages become Gemini role `user`.
- `assistant` messages become Gemini role `model`.
- Bedrock-style `toolUse` blocks become Gemini `function_call` parts.
- Bedrock-style `toolResult` blocks become Gemini `function_response` parts.

In [3]:
def extract_system_and_messages(messages: list[dict[str, Any]]) -> tuple[str | None, list[dict[str, Any]]]:
    system_chunks: list[str] = []
    conversation: list[dict[str, Any]] = []
    for message in messages:
        role = message.get("role", "user")
        content = message.get("content", "")
        if role == "system":
            if isinstance(content, str) and content.strip():
                system_chunks.append(content.strip())
            elif isinstance(content, list):
                for block in content:
                    text = block.get("text") if isinstance(block, dict) else None
                    if isinstance(text, str) and text.strip():
                        system_chunks.append(text.strip())
            continue
        conversation.append(message)
    return "\n\n".join(system_chunks) or None, conversation


def text_from_tool_result_content(content: Any) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        chunks: list[str] = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                chunks.append(str(item["text"]))
            else:
                chunks.append(json.dumps(item, ensure_ascii=True))
        return "\n".join(chunks)
    return json.dumps(content, ensure_ascii=True)


def response_payload_from_tool_result(tool_result: dict[str, Any]) -> dict[str, Any]:
    text = text_from_tool_result_content(tool_result.get("content", []))
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed
        return {"result": parsed}
    except json.JSONDecodeError:
        return {"result": text}


def convert_messages_to_gemini(messages: list[dict[str, Any]]) -> tuple[str | None, list[types.Content]]:
    system_instruction, conversation = extract_system_and_messages(messages)
    tool_names_by_id: dict[str, str] = {}
    contents: list[types.Content] = []

    for message in conversation:
        role = message.get("role", "user")
        gemini_role = "model" if role == "assistant" else "user"
        raw_content = message.get("content", "")
        parts: list[types.Part] = []

        if isinstance(raw_content, str):
            parts.append(types.Part.from_text(text=raw_content))
        elif isinstance(raw_content, list):
            for block in raw_content:
                if not isinstance(block, dict):
                    parts.append(types.Part.from_text(text=str(block)))
                    continue
                if "text" in block:
                    parts.append(types.Part.from_text(text=str(block["text"])))
                    continue
                tool_use = block.get("toolUse")
                if isinstance(tool_use, dict):
                    name = str(tool_use.get("name") or "")
                    args = tool_use.get("input") if isinstance(tool_use.get("input"), dict) else {}
                    tool_use_id = str(tool_use.get("toolUseId") or tool_use.get("id") or name)
                    if tool_use_id and name:
                        tool_names_by_id[tool_use_id] = name
                    part = types.Part.from_function_call(name=name, args=args)
                    thought_signature = tool_use.get("thoughtSignature") or tool_use.get("thought_signature")
                    if isinstance(thought_signature, bytes):
                        part.thought_signature = thought_signature
                    parts.append(part)
                    continue
                tool_result = block.get("toolResult")
                if isinstance(tool_result, dict):
                    tool_use_id = str(tool_result.get("toolUseId") or tool_result.get("id") or "")
                    name = tool_names_by_id.get(tool_use_id) or str(tool_result.get("name") or "")
                    if not name:
                        raise RuntimeError(f"Cannot map Gemini tool result without a tool name: {tool_result}")
                    parts.append(types.Part.from_function_response(name=name, response=response_payload_from_tool_result(tool_result)))
                    continue
                parts.append(types.Part.from_text(text=json.dumps(block, ensure_ascii=True)))
        else:
            parts.append(types.Part.from_text(text=json.dumps(raw_content, ensure_ascii=True)))

        if parts:
            contents.append(types.Content(role=gemini_role, parts=parts))

    return system_instruction, contents


## Step 3: Convert Generic Tools

The agent loop may pass Anthropic/OpenAI-style function tools or custom tool descriptors. The production Gemini provider should normalize both shapes into `types.Tool(function_declarations=[...])`.

In [4]:
def normalize_function_tool(tool: dict[str, Any]) -> dict[str, Any] | None:
    if tool.get("type") == "function":
        name = tool.get("name")
        parameters = tool.get("parameters") or {"type": "object", "properties": {}}
        description = tool.get("description", "")
    else:
        custom = tool.get("custom") or {}
        name = custom.get("name") or tool.get("name")
        parameters = (
            custom.get("input_schema")
            or custom.get("inputSchema")
            or custom.get("parameters")
            or tool.get("input_schema")
            or tool.get("inputSchema")
            or tool.get("parameters")
            or {"type": "object", "properties": {}}
        )
        description = custom.get("description") or tool.get("description", "")

    if not name:
        return None
    return {"name": name, "description": description, "parameters": parameters}


def convert_tools_to_gemini(tools: list[dict[str, Any]] | None) -> list[types.Tool] | None:
    declarations: list[types.FunctionDeclaration] = []
    for tool in tools or []:
        normalized = normalize_function_tool(tool)
        if not normalized:
            continue
        declarations.append(
            types.FunctionDeclaration(
                name=normalized["name"],
                description=normalized["description"],
                parameters_json_schema=normalized["parameters"],
            )
        )
    if not declarations:
        return None
    return [types.Tool(function_declarations=declarations)]


## Step 4: Prototype `stream_response()`

This function mirrors the future provider method. It yields only the generic event types that the rest of the app understands.

In [5]:
def first_candidate_parts(chunk: types.GenerateContentResponse) -> list[types.Part]:
    if not chunk.candidates:
        return []
    content = chunk.candidates[0].content
    if not content or not content.parts:
        return []
    return list(content.parts)


async def stream_gemini_response(
    messages: list[dict[str, Any]],
    credentials: dict[str, str],
    tools: list[dict[str, Any]] | None = None,
    model: str | None = None,
) -> AsyncIterator[LLMEvent]:
    model_id = model or DEFAULT_MODEL
    local_client = build_gemini_client(credentials)
    system_instruction, contents = convert_messages_to_gemini(messages)
    gemini_tools = convert_tools_to_gemini(tools)
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=gemini_tools,
        max_output_tokens=4096,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    )

    seen_tool_calls: set[str] = set()
    last_finish_reason = "completed"
    stream = await local_client.aio.models.generate_content_stream(
        model=model_id,
        contents=contents,
        config=config,
    )
    async for chunk in stream:
        if chunk.candidates and chunk.candidates[0].finish_reason:
            last_finish_reason = str(chunk.candidates[0].finish_reason)
        for part in first_candidate_parts(chunk):
            if part.text:
                yield LLMEvent(type="text", content=part.text)
            if part.function_call:
                function_call = part.function_call
                tool_name = function_call.name or ""
                tool_input = dict(function_call.args or {})
                tool_use_id = function_call.id or f"gemini-{tool_name}-{json.dumps(tool_input, sort_keys=True)}"
                if tool_use_id in seen_tool_calls:
                    continue
                seen_tool_calls.add(tool_use_id)
                yield LLMEvent(
                    type="tool_use",
                    tool_use_id=tool_use_id,
                    tool_name=tool_name,
                    tool_input=tool_input,
                    thought_signature=part.thought_signature,
                )

    yield LLMEvent(type="stop", content=last_finish_reason)


async def collect_events(*args: Any, **kwargs: Any) -> list[LLMEvent]:
    events = []
    async for event in stream_gemini_response(*args, **kwargs):
        events.append(event)
        print(asdict(event))
    return events


## Step 5: Verify Plain Text Streaming

Expected result: multiple `text` events followed by one `stop` event.

In [6]:
plain_messages = [
    {"role": "system", "content": "You are concise and factual."},
    {"role": "user", "content": "Reply with exactly one sentence explaining why provider interfaces are useful."},
]

plain_events = await collect_events(plain_messages, notebook_credentials)


{'type': 'text', 'content': "Provider interfaces decouple a service's specification from its concrete implementation, enabling developers to easily", 'tool_use_id': '', 'tool_name': '', 'tool_input': {}, 'thought_signature': None}
{'type': 'text', 'content': ' swap, mock, or extend functionality without modifying the core codebase.', 'tool_use_id': '', 'tool_name': '', 'tool_input': {}, 'thought_signature': None}
{'type': 'stop', 'content': 'FinishReason.STOP', 'tool_use_id': '', 'tool_name': '', 'tool_input': {}, 'thought_signature': None}


## Step 6: Verify Function Calling

Expected result: Gemini emits a `tool_use` event for `get_account_status`. In production, `src.agents.agentic_loop` will execute the tool, append the `toolUse` and `toolResult` blocks, and call the provider again.

In [7]:
sample_tools = [
    {
        "type": "function",
        "name": "get_account_status",
        "description": "Return the account status for a user by email address.",
        "parameters": {
            "type": "object",
            "properties": {
                "email": {"type": "string", "description": "User email address"}
            },
            "required": ["email"],
            "additionalProperties": False,
        },
    }
]

tool_messages = [
    {"role": "system", "content": "Use tools when account data is needed. Do not invent account state."},
    {"role": "user", "content": "What is the account status for alex@example.com?"},
]

tool_events = await collect_events(tool_messages, notebook_credentials, sample_tools)


{'type': 'tool_use', 'content': '', 'tool_use_id': 'call_3475447', 'tool_name': 'get_account_status', 'tool_input': {'email': 'alex@example.com'}, 'thought_signature': b'\x12\xa7\x03\n\xa4\x03\x01\x11M2\x0f\xc6%/\x14]N\x05BBPy\xbd{o\xff\xd8\xb7jW\xaa\xdct\xc3:\xbe\xa9\xa8\xab\xae\xe4[#D\x97Y\x89\xd1\x9e\xae\x8d\xce\x01\x08\xde\xf9\xe3\xc0\x8c\x90\x84\xcf\xc3\xcb\x0cRRB\x1411\xf0\xca\x11W;\xf1\xc3\xda.\xb3RD\n\xd1&\xb2\xcb\xe2\xda:r&\xa7\xb9\xd0~$\xd5\xe6\xff\x87\xdf\x9f\xe3\xd8}\x08\x15>@\xd3\x1fw\\\xa0MWbCf\xf4J\xac\r\x89\xbc\xc9o\x10\x05\x08\x89\x0b\xbcM\\\x83\xe4\x94<\x99)\x910\x07\x14\xf3t\x8cX\xbb\xb5\xb11X\x94\xc2\xe3\xa7fy\xecs3\xd0\xae\xdb\xc7\xc3\xe65>\',\xdf4$\x8c=JI\x97NT \xac\x1c\x00\xa4\xec\x95\xf5\xcc-x\xf8l\'\xc6\xf2\x0fAN\xdd\x80uL\xf6n\\\x93Q\xb69\xbd\xb6?e\x8f\x89Z\x18p\xf2\x88)\xce\x86Y\x14+U\xc1\x01\xeav\x1a;\x19\xa4\x8b\xb60\x80\x8c\xf9q|QD\x8fO:\r\x9c\xa6\xb9\xc1&\xb9\xa4\xc8\x84_\xa4\x7f\xa7\x06\x93\xb3\x0ft\x07\x19&\xa9U\x8d\x8eD\x1f\'\xd4\xba\xa49\xc3\xd2\xfe\x

## Step 7: Verify Tool Result Round Trip

This cell simulates the agent loop after a tool call. It appends an assistant `toolUse` block and a user `toolResult` block using the event emitted by Gemini, then asks Gemini to produce the final user-facing answer.

In [8]:
tool_use_events = [event for event in tool_events if event.type == "tool_use"]
if not tool_use_events:
    raise RuntimeError("Run the previous cell until Gemini emits a tool_use event.")

tool_event = tool_use_events[0]
thought_signature = getattr(tool_event, "thought_signature", None)
if thought_signature is None:
    raise RuntimeError(
        "The Gemini tool event is missing thought_signature. Restart the notebook kernel "
        "and rerun the setup, streaming prototype, and function-calling cells before Step 7."
    )
messages_with_tool_result = [
    *tool_messages,
    {
        "role": "assistant",
        "content": [
            {
                "toolUse": {
                    "toolUseId": tool_event.tool_use_id,
                    "name": tool_event.tool_name,
                    "input": tool_event.tool_input,
                    "thoughtSignature": thought_signature,
                }
            }
        ],
    },
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_event.tool_use_id,
                    "content": [
                        {"text": json.dumps({"email": "alex@example.com", "status": "active", "plan": "pro"})}
                    ],
                }
            }
        ],
    },
]

final_events = await collect_events(messages_with_tool_result, notebook_credentials, sample_tools)


{'type': 'text', 'content': 'The account', 'tool_use_id': '', 'tool_name': '', 'tool_input': {}, 'thought_signature': None}
{'type': 'text', 'content': ' status for alex@example.com is active, and they are on the pro', 'tool_use_id': '', 'tool_name': '', 'tool_input': {}, 'thought_signature': None}
{'type': 'text', 'content': ' plan.', 'tool_use_id': '', 'tool_name': '', 'tool_input': {}, 'thought_signature': None}
{'type': 'stop', 'content': 'FinishReason.STOP', 'tool_use_id': '', 'tool_name': '', 'tool_input': {}, 'thought_signature': None}


## Production Implementation Notes

If the cells above pass, implement `api/src/llm/providers/gemini.py` using these decisions:

- Class name: `GeminiProvider(LLMProvider)`.
- Default model: start with `gemini-2.5-flash`, with per-agent `model` override support.
- Credentials: require `api_key` for the first production version unless OAuth storage is explicitly added to provider settings.
- Message conversion: reuse `system_instruction`, `user` -> `user`, `assistant` -> `model`.
- Tool declarations: normalize `type=function` and `custom` tools into Gemini `FunctionDeclaration(parameters_json_schema=...)`.
- Text streaming: emit `LLMEvent(type="text", content=part.text)`.
- Tool calls: emit `LLMEvent(type="tool_use", tool_use_id=..., tool_name=..., tool_input=...)`.
- Stop: emit one `LLMEvent(type="stop", content=<finish_reason or completed>)`.
- Factory: add `"Gemini": GeminiProvider()` to `api/src/llm/providers/factory.py`.
- Models: add a Gemini model list in `api/src/llm/models.py`, preferably settings-driven like OpenAI.
- Tests: add unit tests for message conversion, tool normalization, and event extraction without making real network calls.